# LangGraph와 Llama Stack으로 에이전트 만들기

지난 노트북에서는 ReAct 에이전트를 **처음부터 직접** 만들었습니다. 다음과 같은 코드를 작성했습니다.
- 정규식을 이용한 수동 응답 파싱
- 직접 만든 반복 루프
- 상태 관리
- 도구 실행 로직
- 오류 처리

코드가 200줄이 넘었습니다. 이제 **LangGraph**가 이 작업을 얼마나 간단하게 만들어 주는지 살펴보겠습니다.

## 만들게 될 것

다음 기능을 갖춘 지식 기반 어시스턴트입니다.
1. 문서에서 답을 검색합니다(RAG)
2. 답할 수 없으면 교수와의 미팅 일정을 잡습니다

기능은 같고, 코드는 훨씬 적습니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from llama_stack_client import LlamaStackClient
import json

In [ ]:
#일부 로그 비활성화
import logging

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)

## Llama Stack에 연결하기

LangGraph는 MCP 도구 직접 바인딩 등 확장된 기능을 제공하는 **Responses API**를 사용해 Llama Stack에 연결합니다.

**중요**: `use_responses_api=True`를 사용할 때 base URL은 다음과 같아야 합니다.
- ✅ `http://llama-stack-service:8321/v1` (Responses API 엔드포인트)
- ❌ 사용하지 말 것: `http://llama-stack-service:8321/v1/openai/v1` (이것은 표준 Chat Completions용입니다)

매끄러운 MCP 통합을 가능하게 하는 것이 바로 Responses API입니다!

In [ ]:
# LangGraph는 Llama Stack Responses API를 통해 연결합니다
# use_responses_api=True를 사용할 때 base는 /v1이어야 합니다(/v1/openai/v1이 아님)
# Responses API는 MCP 통합을 위한 도구 바인딩을 지원합니다
llm = ChatOpenAI(
    openai_api_base="http://llama-stack-service:8321/v1",
    model="vllm-llama32/llama32",
    openai_api_key="not-needed",
    use_responses_api=True,
    temperature=0.1
)

# RAG용 표준 클라이언트도 연결
llama_client = LlamaStackClient(base_url="http://llama-stack-service:8321")

print(f"✅ Connected to Llama Stack with Responses API enabled")

## 도구 정의하기

에이전트가 다음 도구를 사용할 수 있게 합니다.
1. **RAG 검색** - 지식 베이스에서 정보 검색
2. **교수 디렉터리** - 전문 분야로 교수 찾기  
3. **MCP 캘린더 도구** - MCP 서버 직접 통합!

MCP 캘린더 통합에는 **직접 도구 바인딩**을 지원하는 Llama Stack의 Responses API를 사용합니다. 도구 목록에 MCP 바인딩만 넣으면 나머지는 LangGraph가 모두 자동으로 처리합니다.

- MCP 바인딩은 앞에서 설정한 실제 캘린더 서버에 연결됩니다
- 9개의 캘린더 도구(create_event, get_upcoming_events, search_events 등)를 모두 사용할 수 있게 됩니다
- 도구 호출은 Llama Stack을 통해 비동기로 처리됩니다
- 래퍼 함수를 직접 만들 필요가 없습니다!

이전 노트북(`2-mcp-servers.ipynb`, `3-agentic-workflows.ipynb`)에서 사용한 것과 같은 MCP 캘린더 서버입니다.

In [ ]:
# 먼저 RAG에 사용할 vector store를 가져옵니다
try:
    vector_stores = llama_client.vector_stores.list()
    
    if vector_stores.data and len(vector_stores.data) > 0:
        vector_store = vector_stores.data[0]
        print(f"✅ Using vector store: {vector_store.id}")
    else:
        raise Exception("Can't find an existing vector store")
    
    # 정상 상태인지 확인하기 위한 테스트 질의
    query = "canopy"
    max_results = 5
    search_results = llama_client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=query,
        max_num_results=max_results,
        search_mode="vector"
    )
    print(f"search_results: {search_results.data}")

    if vector_store.file_counts.total > 0 and not search_results.data:
        llama_client.vector_stores.delete(vector_store_id=vector_store.id)
        vector_store = llama_client.vector_stores.create(
            name="my_citations_db",
            extra_body={
                "embedding_model": "sentence-transformers/nomic-ai/nomic-embed-text-v1.5",
                "embedding_dimension": 384,
                "provider_id": "milvus",
                "vector_db_id": "test"
            }
        )

except Exception as e:
    print(f"⚠️  Vector store setup failed: {e}")
    vector_store = None

In [ ]:
# 교수 디렉터리 - 실제 시스템에서는 데이터베이스가 됩니다
PROFESSORS = {
    "Dr. Sarah Chen": {
        "department": "Computer Science",
        "expertise": ["Machine Learning", "Neural Networks", "AI Ethics", "Agentic Workflows"],
        "email": "s.chen@university.edu"
    },
    "Prof. Michael Rodriguez": {
        "department": "Physics",
        "expertise": ["Quantum Mechanics", "Particle Physics", "Quantum Chromodynamics"],
        "email": "m.rodriguez@university.edu"
    },
    "Dr. Emily Thompson": {
        "department": "Biology",
        "expertise": ["Botany", "Ecology", "Forest Canopy Structure", "Plant Biology"],
        "email": "e.thompson@university.edu"
    },
    "Prof. James Wilson": {
        "department": "Computer Science",
        "expertise": ["Distributed Systems", "Cloud Computing", "Software Architecture"],
        "email": "j.wilson@university.edu"
    }
}


@tool
def search_knowledge_base(query: str) -> str:
    """Search through documents to find information. Use this when the user asks about concepts, definitions, or topics."""
    if not vector_store:
        return "Error: Knowledge base not available. Please set up the vector store first."
    
    try:
        results = llama_client.vector_stores.search(
            vector_store_id=vector_store.id,
            query=query,
            max_num_results=3,
            search_mode="vector"
        )
        
        if not results.data:
            return "No relevant information found in the knowledge base."
        
        formatted_results = []
        for i, result in enumerate(results.data, 1):
            content = result.content if hasattr(result, 'content') else str(result)
            formatted_results.append(f"Result {i}: {content}")
        
        return "\n\n".join(formatted_results)
    except Exception as e:
        return f"Error searching knowledge base: {str(e)}"


@tool
def find_professors_by_expertise(topic: str) -> str:
    """Find professors who have expertise in a specific topic or subject area.
    
    Args:
        topic: The topic or subject area to search for (e.g., 'Machine Learning', 'Quantum Physics', 'Botany')
    
    Returns:
        List of professors with matching expertise, including their name, department, and contact info
    """
    matching_profs = []
    
    for name, info in PROFESSORS.items():
        # 주제가 교수의 전문 분야 중 하나와 일치하는지 확인
        if any(topic.lower() in exp.lower() or exp.lower() in topic.lower() for exp in info["expertise"]):
            matching_profs.append((name, info))
    
    if not matching_profs:
        # 정확히 일치하는 교수가 없으면 모든 교수를 표시
        result = f"No professors found with specific expertise in '{topic}'.\n\n"
        result += "Available professors:\n\n"
        for name, info in PROFESSORS.items():
            result += f"**{name}** - {info['department']}\n"
            result += f"  Expertise: {', '.join(info['expertise'])}\n"
            result += f"  Email: {info['email']}\n\n"
        return result
    
    result = f"Professors with expertise in '{topic}':\n\n"
    for name, info in matching_profs:
        result += f"**{name}** - {info['department']}\n"
        result += f"  Expertise: {', '.join(info['expertise'])}\n"
        result += f"  Email: {info['email']}\n\n"
    
    return result


# 전체 도구 정의: 커스텀 도구 + MCP 바인딩
tools = [
    search_knowledge_base,
    find_professors_by_expertise,
    # MCP 캘린더 서버 바인딩
    {
        "type": "mcp",
        "server_label": "canopy-calendar",
        "server_url": "http://canopy-mcp-calendar-mcp-server:8080/sse",
        "require_approval": "never",
    }
]

print("✅ Tools defined:")
print("   Custom tools:")
print("     - search_knowledge_base")
print("     - find_professors_by_expertise")
print("   MCP calendar server:")
print("     - canopy-calendar (provides 9 calendar tools via Responses API)")

## 에이전트 만들기

여기서 마법이 일어납니다. LangGraph를 사용하면 ReAct 에이전트를 **코드 한 줄**로 만들 수 있습니다.

파싱도, 루프도, 상태 관리도 필요 없습니다. LangGraph가 모두 처리합니다.

**내부에서 일어나는 일:**
- `tools` 목록에 MCP 캘린더 바인딩을 전달합니다
- `create_react_agent()`가 모든 도구(커스텀 + MCP)를 LLM에 자동으로 바인딩합니다
- Responses API가 MCP 도구 바인딩을 지원하므로 MCP 서버와 직접 통합할 수 있습니다
- LangGraph가 모든 비동기 MCP 함수 호출을 자동으로 처리합니다
- 이제 에이전트가 실제 캘린더 데이터베이스와 상호작용할 수 있습니다!

In [ ]:
from datetime import datetime

# 자동 일정 관리 에이전트용 시스템 프롬프트
system_prompt = f"""You are a helpful assistant that helps students with their calendar and studies.
Today is {datetime.today().strftime('%Y-%m-%d')}.

Your workflow:

1. If student asks about their schedule ("What lectures do I have?"):
   - Call get_upcoming_events
   - Show them the results
   - DONE (don't modify anything)

2. If student asks a question about a topic ("I need help understanding X"):
   - First: call search_knowledge_base with the topic
   - If knowledge base has relevant information: answer their question with that information, DONE
   - If knowledge base has NO relevant information:
     a) Call find_professors_by_expertise to find an expert
     b) Call get_events_by_date to check for scheduling conflicts
     c) Call create_event to schedule a meeting with the professor at a free time
     d) Tell the student you scheduled the meeting

When scheduling with create_event:
- Pick a reasonable time that's free (check with get_events_by_date first)
- Use these parameters: name, category, level, start_time, end_time, content
- Do NOT include sid, status, or creation_time
"""

In [ ]:
# 모든 도구를 사용하는 ReAct 에이전트 생성
agent = create_react_agent(
    llm,
    tools,
    prompt=system_prompt,
    checkpointer=MemorySaver(),
)

print("✅ ReAct agent created!")
print("\n🎯 The agent has access to:")
print("   • 2 custom tools (RAG search, professor directory)")
print("   • 9 MCP calendar tools (automatically bound via Responses API)")
print("\n🎯 Compare this to the 200+ lines you wrote before...")

에이전트를 간단한 다이어그램으로도 볼 수 있습니다.

In [ ]:
agent

### 방금 무슨 일이 일어났나요?

함수 호출 하나로 이전 노트북의 다음 코드가 모두 대체되었습니다.
- ❌ `parse_react_response()` - 정규식을 이용한 수동 파싱
- ❌ `execute_tool()` - 도구 라우팅 로직
- ❌ `run_react_agent()` - 전체 반복 루프
- ❌ 대화 기록 관리
- ❌ 오류 처리와 재시도 로직
- ❌ 종료 조건 검사

LangGraph가 이 **모든 작업을 자동으로** 처리합니다.

## 에이전트 테스트하기

실제로 동작하는 모습을 확인해 봅시다!

먼저 에이전트의 출력을 보여 주는 로직을 추가한 뒤, 에이전트를 실행해 어떤 결과가 나오는지 확인합니다.

In [ ]:
import uuid
from textwrap import indent
from langchain_core.messages import HumanMessage
import pprint

def run_agent(question: str, thread_id: str | None = None):
    """LangGraph 에이전트를 실행하고 값을 스트리밍하며, MCP 도구 호출과 최종 답변을 표시합니다."""
    thread_id = thread_id or str(uuid.uuid4())

    print("\n" + "="*80)
    print("USER INPUT")
    print("="*80 + "\n")
    print(question, "\n")

    config = {"configurable": {"thread_id": thread_id}}
    inputs = {"messages": [HumanMessage(content=question)]}

    seen_messages = 0
    final_state = None
    tool_header_printed = False

    # 중요: stream_mode="values"
    for state in agent.stream(inputs, config, stream_mode="values"):

        messages = state.get("messages", [])
        new_messages = messages[seen_messages:]
        seen_messages = len(messages)

        for msg in new_messages:
            msg_type = getattr(msg, "type", None)

            if msg_type == "ai":
                if getattr(msg, "tool_calls", None):
                    if not tool_header_printed:
                        print("="*80)
                        print("TOOL CALLS")
                        print("="*80 + "\n")
                        tool_header_printed = True

                    for tc in msg.tool_calls:
                        name = tc.get("name")
                        args = tc.get("args")
                        print(f"🔧 Calling tool: {name}")
                        print(f"   Args: {args}\n")
                else:
                    tool_outputs = msg.additional_kwargs.get("tool_outputs", [])

                    # MCP 도구 호출 정보는 여기에 있음
                    for t in tool_outputs:
                        if t.get("type") == "mcp_call":
                            if not tool_header_printed:
                                print("="*80)
                                print("TOOL CALLS")
                                print("="*80 + "\n")
                                tool_header_printed = True

                            name = t.get("name")
                            server = t.get("server_label")
                            args = t.get("arguments")
                            error = t.get("error", "")
                            output = t.get("output", "")

                            print(f"🔧 MCP TOOL CALL: {name}  (server: {server})")
                            print(f"   args: {args}")
                            if output:
                                print("   output:")
                                print(indent(str(output), "      "))
                            elif error:
                                print("   error:")
                                print(indent(str(error), "      "))
                            print()

            elif msg_type == "tool":
                # 결과를 얼마나 보여 줄지 원하는 대로 조정할 수 있음
                print(f"📦 Tool result ({msg.name}): {str(msg.content)[:200]}...\n")

        final_state = state

    if final_state is None:
        print("⚠️ Agent produced no state")
        return

    # 최종 어시스턴트 메시지 = 마지막 AI 메시지
    messages = final_state.get("messages", [])
    final_answer = None
    for msg in reversed(messages):
        if getattr(msg, "type", None) == "ai":
            final_answer = msg
            break

    print("="*80)
    print("FINAL AGENT RESPONSE")
    print("="*80 + "\n")
    if final_answer:
        content = final_answer.content
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and "text" in item:
                    print(item["text"])
                elif hasattr(item, "text"):
                    print(item.text)
        else:
            print(content)
    else:
        print("(No assistant answer found)")
    print("="*80 + "\n")


### 예제 1: 지식 베이스 검색

에이전트는 이 정보를 지식 베이스에서 검색해야 합니다.

**참고**: vector store에 문서가 있어야 합니다. RAG 노트북(`5-rag/2-intro-to-RAG.ipynb`)을 진행하지 않았다면 지식 베이스가 비어 있습니다.

In [ ]:
run_agent("What is the structure of a forest canopy in botany?")

### 예제 2: 전문가 찾기와 일정 잡기

에이전트가 질문에 답할 수 없을 때 다음을 수행하는지 지켜보세요.
1. 관련 전문 분야의 교수 검색
2. 캘린더에 미팅 일정 등록

In [ ]:
run_agent("I need help understanding quantum chromodynamics. Can you find me an expert and schedule a meeting for February 18th, 2026 at 4pm for one hour?")

### 예제 3: MCP 캘린더 사용하기

에이전트가 실제 캘린더를 확인하고 일정을 검색할 수 있습니다!

In [ ]:
run_agent("What upcoming lectures do I have?")

## 직접 해 보세요!

이제 실험해 보세요.
1. 지식 베이스 검색이 필요한 질문하기
2. 미팅 일정 등록이 필요한 질문하기
3. 에이전트가 두 도구를 차례로 사용하도록 만들어 보기

에이전트가 어떤 도구를 사용할지 스스로 추론하는 과정을 확인해 보세요!

In [ ]:
# 여러분 차례입니다!
run_agent("YOUR QUESTION HERE")